# Qwen3-0.6B: arsitektur dan layer skipping

Notebook ini memuat model, menginspeksi modul internal, lalu mendemonstrasikan pengaktifan dan skipping layer menurut rancangan proposal: 28 layer dibagi menjadi tujuh kelompok berurutan (empat layer per kelompok); G1 dan G7 selalu aktif. Route demonstrasi dipilih manual, bukan oleh router terlatih. Ini bukan evaluasi kualitas atau kecepatan.

Pemuatan pertama mengunduh bobot sekitar 1,5 GB. CPU FP32 memerlukan sekitar 3 GB RAM untuk bobot saja, ditambah memori kerja.

## 1. Muat model
Versi library mengikuti `experiment/requirements.txt`. Contoh memakai CPU FP32 seperti lingkungan awal penelitian.

In [ ]:
from pathlib import Path
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-0.6B'
PROJECT_DIR = Path.cwd() / 'experiment' if (Path.cwd() / 'experiment' / 'requirements.txt').exists() else Path.cwd()
CACHE_DIR = PROJECT_DIR / '.cache' / 'huggingface'
DEVICE = torch.device('cpu')
torch.set_num_threads(4)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR, torch_dtype=torch.float32, low_cpu_mem_usage=True).to(DEVICE)
model.eval()
print(f'Transformers {transformers.__version__} | {DEVICE} | {next(model.parameters()).dtype}')

## 2. Struktur Qwen3-0.6B

Qwen3-0.6B adalah model bahasa **decoder-only**: token masuk dipetakan ke vektor, diproses berurutan oleh 28 decoder layer, lalu diproyeksikan menjadi skor token berikutnya. Setiap layer memakai pre-normalization RMSNorm, causal Grouped-Query Attention (GQA), dan MLP SwiGLU. Di dalam attention, Q dan K dinormalisasi per-head sebelum RoPE memberi informasi posisi.

Diagram di bawah dibuat dari konfigurasi dan modul model yang sedang dimuat, jadi ukuran pentingnya tidak diketik ulang secara manual. Diagram kedua memperbesar satu decoder layer dan menunjukkan dua jalur residual. Untuk rancangan skipping, keluaran hidden state dapat diteruskan melewati satu kelompok layer; embedding, final norm, dan LM head tetap berada di jalur model. G1 dan G7 ditandai sebagai kelompok yang selalu aktif sesuai rancangan penelitian.

Rujukan: [konfigurasi resmi Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B/blob/main/config.json) dan [implementasi Qwen3 pada Transformers 4.51.3](https://github.com/huggingface/transformers/blob/v4.51.3/src/transformers/models/qwen3/modeling_qwen3.py). Batas konteks yang didukung konfigurasi model tidak sama dengan batas eksperimen; rancangan eksperimen saat ini membatasi prompt hingga 3.840 token dan total konteks hingga 4.096 token.


In [ ]:
from IPython.display import HTML, display

config = model.config
base = model.model
layers = base.layers


def parameter_count(module):
    return sum(parameter.numel() for parameter in module.parameters())


layer_count = len(layers)
hidden_size = config.hidden_size
intermediate_size = config.intermediate_size
query_heads = config.num_attention_heads
kv_heads = config.num_key_value_heads
head_dim = getattr(config, "head_dim", hidden_size // query_heads)
vocab_size = config.vocab_size
group_size = 4
group_count = layer_count // group_size
assert layer_count % group_size == 0, "Jumlah layer harus habis dibagi ukuran kelompok"

print("Ringkasan konfigurasi yang sedang dimuat")
print(f"Model                 : {config.model_type} / {model.__class__.__name__}")
print(f"Jumlah parameter      : {parameter_count(model):,}")
print(f"Vocabulary            : {vocab_size:,} token")
print(f"Hidden size           : {hidden_size}")
print(f"Decoder layer         : {layer_count}")
print(f"Attention head        : {query_heads} query, {kv_heads} key/value")
print(f"Dimensi per head      : {head_dim}")
print(f"Ukuran MLP            : {intermediate_size}")
print(f"Konteks konfigurasi   : {config.max_position_embeddings:,} posisi maksimum")
print(f"Embedding             : {parameter_count(base.embed_tokens):,} parameter")
print(f"Satu decoder layer    : {parameter_count(layers[0]):,} parameter")
print(f"Final RMSNorm         : {parameter_count(base.norm):,} parameter")
print(f"LM head               : {parameter_count(model.lm_head):,} parameter")
print(f"Embedding/head berbagi: {getattr(config, 'tie_word_embeddings', False)}")

# Diagram 1: alur token melewati keseluruhan model. Setiap blok kecil adalah satu layer.
stack_rows = []
for group_index in range(group_count):
    group_name = f"G{group_index + 1}"
    fixed_group = group_index in (0, group_count - 1)
    fill = "#d9f2e6" if fixed_group else "#e8eef8"
    stroke = "#25845b" if fixed_group else "#637590"
    layer_boxes = []
    for layer_offset in range(group_size):
        layer_number = group_index * group_size + layer_offset + 1
        x = 270 + layer_offset * 82
        y = 80 + group_index * 44
        layer_boxes.append(
            f'<rect x="{x}" y="{y}" width="68" height="30" rx="7" '
            f'fill="{fill}" stroke="{stroke}" stroke-width="1.5"/>'
            f'<text x="{x + 34}" y="{y + 20}" text-anchor="middle" '
            f'font-size="12" fill="#233047">L{layer_number}</text>'
        )
    status = "selalu aktif" if fixed_group else "dapat dirutekan"
    y_text = 98 + group_index * 44
    stack_rows.append(
        f'<text x="40" y="{y_text}" font-size="14" font-weight="700" '
        f'fill="#233047">{group_name}</text>'
        + "".join(layer_boxes)
        + f'<text x="650" y="{y_text}" font-size="12" fill="#526176">{status}</text>'
    )

overview_svg = f'''<svg viewBox="0 0 900 510" role="img" aria-label="Diagram alur arsitektur Qwen3-0.6B" xmlns="http://www.w3.org/2000/svg">
  <defs><marker id="arrow-overview" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M0,0 L0,6 L7,3 z" fill="#526176"/></marker></defs>
  <style>text {{ font-family: Arial, sans-serif; }}</style>
  <text x="450" y="28" text-anchor="middle" font-size="19" font-weight="700" fill="#17243a">Alur pemrosesan token</text>
  <rect x="40" y="45" width="820" height="350" rx="16" fill="#fbfcfe" stroke="#d8dfeb"/>
  <rect x="40" y="80" width="150" height="54" rx="10" fill="#fff1d6" stroke="#c78a20"/>
  <text x="115" y="103" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">ID token</text>
  <text x="115" y="121" text-anchor="middle" font-size="11" fill="#526176">prompt / token baru</text>
  <line x1="190" y1="107" x2="245" y2="107" stroke="#526176" stroke-width="2" marker-end="url(#arrow-overview)"/>
  <rect x="250" y="55" width="420" height="315" rx="12" fill="#ffffff" stroke="#9eabc0" stroke-width="1.5"/>
  <text x="460" y="73" text-anchor="middle" font-size="13" font-weight="700" fill="#233047">{layer_count} decoder layer · {group_count} kelompok × {group_size} layer</text>
  {"".join(stack_rows)}
  <line x1="670" y1="107" x2="710" y2="107" stroke="#526176" stroke-width="2" marker-end="url(#arrow-overview)"/>
  <rect x="715" y="80" width="125" height="54" rx="10" fill="#eee8fa" stroke="#8162b2"/>
  <text x="777" y="103" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">Final RMSNorm</text>
  <text x="777" y="121" text-anchor="middle" font-size="11" fill="#526176">lalu LM head</text>
  <line x1="777" y1="134" x2="777" y2="165" stroke="#526176" stroke-width="2" marker-end="url(#arrow-overview)"/>
  <rect x="715" y="170" width="125" height="58" rx="10" fill="#dfeefa" stroke="#4284a9"/>
  <text x="777" y="194" text-anchor="middle" font-size="13" font-weight="700" fill="#233047">Logit vocabulary</text>
  <text x="777" y="212" text-anchor="middle" font-size="11" fill="#526176">{vocab_size:,} skor token</text>
  <path d="M777 228 V260 H115 V140" fill="none" stroke="#4284a9" stroke-width="2" marker-end="url(#arrow-overview)"/>
  <text x="445" y="253" text-anchor="middle" font-size="11" fill="#4284a9">Pilih token berikutnya, lalu ulangi proses decoding</text>
  <text x="50" y="420" font-size="12" font-weight="700" fill="#25845b">Hijau: G1 dan G{group_count} selalu aktif dalam rancangan skipping</text>
  <text x="50" y="441" font-size="12" fill="#526176">Biru: kelompok tengah dapat aktif atau dilewati sesuai route per artikel</text>
  <text x="50" y="462" font-size="12" fill="#526176">Jika dilewati, hidden state diteruskan ke layer berikutnya; bobot layer tetap ada.</text>
</svg>'''

# Diagram 2: satu decoder layer, termasuk operasi di cabang perhatian dan MLP.
block_svg = f'''<svg viewBox="0 0 900 650" role="img" aria-label="Diagram isi satu decoder layer Qwen3" xmlns="http://www.w3.org/2000/svg">
  <defs><marker id="arrow-block" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto"><path d="M0,0 L0,6 L7,3 z" fill="#526176"/></marker></defs>
  <style>text {{ font-family: Arial, sans-serif; }}</style>
  <text x="450" y="28" text-anchor="middle" font-size="19" font-weight="700" fill="#17243a">Isi satu decoder layer</text>
  <text x="450" y="50" text-anchor="middle" font-size="12" fill="#526176">hidden state masuk dan keluar berdimensi {hidden_size}</text>
  <rect x="345" y="68" width="210" height="42" rx="9" fill="#fff1d6" stroke="#c78a20"/>
  <text x="450" y="94" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">hidden state masuk (x)</text>
  <line x1="450" y1="110" x2="450" y2="143" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)"/>
  <rect x="365" y="148" width="170" height="42" rx="9" fill="#eee8fa" stroke="#8162b2"/>
  <text x="450" y="174" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">Input RMSNorm</text>
  <line x1="450" y1="190" x2="450" y2="220" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)"/>
  <rect x="285" y="225" width="330" height="120" rx="12" fill="#e8f2fb" stroke="#4284a9" stroke-width="1.5"/>
  <text x="450" y="249" text-anchor="middle" font-size="15" font-weight="700" fill="#233047">Causal GQA self-attention</text>
  <text x="450" y="272" text-anchor="middle" font-size="12" fill="#34445b">Q: {query_heads} head · K/V: {kv_heads} head · dimensi head: {head_dim}</text>
  <text x="450" y="293" text-anchor="middle" font-size="12" fill="#34445b">Q/K dinormalkan per-head, lalu diberi informasi posisi dengan RoPE</text>
  <text x="450" y="314" text-anchor="middle" font-size="12" fill="#34445b">Mask kausal; {query_heads // kv_heads} query head berbagi setiap K/V head</text>
  <path d="M345 89 H220 V365 H420" fill="none" stroke="#25845b" stroke-width="2" marker-end="url(#arrow-block)"/>
  <circle cx="450" cy="375" r="20" fill="#d9f2e6" stroke="#25845b" stroke-width="1.5"/>
  <text x="450" y="380" text-anchor="middle" font-size="15" font-weight="700" fill="#233047">+</text>
  <line x1="450" y1="345" x2="450" y2="355" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)" stroke="#526176" stroke-width="2"/>
  <line x1="450" y1="395" x2="450" y2="410" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)"/>
  <text x="196" y="237" text-anchor="middle" font-size="11" fill="#25845b">jalur residual</text>
  <rect x="365" y="415" width="170" height="42" rx="9" fill="#eee8fa" stroke="#8162b2"/>
  <text x="450" y="441" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">Post-attention RMSNorm</text>
  <line x1="450" y1="457" x2="450" y2="480" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)"/>
  <rect x="285" y="485" width="330" height="65" rx="12" fill="#fce9e7" stroke="#bd665b" stroke-width="1.5"/>
  <text x="450" y="508" text-anchor="middle" font-size="14" font-weight="700" fill="#233047">MLP SwiGLU</text>
  <text x="450" y="530" text-anchor="middle" font-size="12" fill="#34445b">gate/up: {hidden_size} → {intermediate_size} → down: {hidden_size}</text>
  <path d="M470 395 H700 V578 H470" fill="none" stroke="#25845b" stroke-width="2" marker-end="url(#arrow-block)"/>
  <circle cx="450" cy="578" r="20" fill="#d9f2e6" stroke="#25845b" stroke-width="1.5"/>
  <text x="450" y="583" text-anchor="middle" font-size="15" font-weight="700" fill="#233047">+</text>
  <line x1="450" y1="550" x2="450" y2="558" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)" stroke="#526176" stroke-width="2"/>
  <line x1="450" y1="598" x2="450" y2="620" stroke="#526176" stroke-width="2" marker-end="url(#arrow-block)"/>
  <text x="720" y="320" font-size="11" fill="#25845b">jalur residual</text>
  <text x="450" y="643" text-anchor="middle" font-size="11" fill="#526176">Silu(gate) × up, lalu proyeksi down; hasil dijumlahkan dengan residual kedua.</text>
</svg>'''

display(HTML(
    "<style>.qwen-viz svg { width: 100%; height: auto; } "
    ".qwen-viz { max-width: 980px; margin: 1rem auto; }</style>"
    f'<div class="qwen-viz">{overview_svg}{block_svg}</div>'
))


In [ ]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())

for label, component in [('model', model), ('embedding', base.embed_tokens), ('satu decoder layer', layers[0]), ('norm akhir', base.norm), ('lm_head', model.lm_head)]:
    print(f'{label}: {count_parameters(component):,} parameter')

## 3. Pemetaan tujuh kelompok proposal
Indeks Python mulai dari nol: G1 adalah indeks 0–3 (layer 1–4), G7 indeks 24–27 (layer 25–28). Lima nilai route mengatur G2 sampai G6.

In [ ]:
GROUP_SIZE = 4
assert len(layers) == 28, f'Diharapkan 28 layer, ditemukan {len(layers)}'
groups = {f'G{i + 1}': list(range(i * GROUP_SIZE, (i + 1) * GROUP_SIZE)) for i in range(7)}
for name, indices in groups.items():
    status = 'selalu aktif' if name in ('G1', 'G7') else 'dipilih route'
    print(f'{name}: layer {indices[0] + 1}–{indices[-1] + 1} ({status})')

## 4. Melewati layer
Transformers 4.51.x mengharapkan layer Qwen3 mengembalikan tuple dengan hidden state sebagai elemen pertama. Pengganti pass-through berikut melewatkan perhitungan layer. Modul asli dipulihkan setelah blok `with`. Ini memakai detail internal library; periksa ulang bila versi Transformers berubah.

In [ ]:
from contextlib import contextmanager
from torch import nn

class SkipDecoderLayer(nn.Module):
    def forward(self, hidden_states, *args, **kwargs):
        return (hidden_states,)

@contextmanager
def temporarily_skip_layers(layer_indices):
    original = {index: layers[index] for index in layer_indices}
    try:
        for index in layer_indices:
            layers[index] = SkipDecoderLayer()
        yield
    finally:
        for index, layer in original.items():
            layers[index] = layer

def active_layers_for_route(middle_route):
    if len(middle_route) != 5 or any(bit not in (0, 1) for bit in middle_route):
        raise ValueError('middle_route needs five binary values for G2–G6')
    names = ['G1'] + [f'G{i}' for i, bit in zip(range(2, 7), middle_route) if bit] + ['G7']
    return [index for name in names for index in groups[name]]

def skipped_layers(active_indices):
    active = set(active_indices)
    return [index for index in range(len(layers)) if index not in active]

## 5. Route manual, inverse, dan generasi
Route awal `(1, 0, 1, 0, 1)` berarti G2, G4, G6 aktif dan G3, G5 dilewati. Route inverse membalik lima bit menjadi `(0, 1, 0, 1, 0)`, jadi G3 dan G5 aktif, sementara G2, G4, dan G6 dilewati. G1 serta G7 tetap aktif di kedua route. Keduanya dijalankan dengan prompt dan greedy decoding yang sama. Pada proposal, router akan memilih satu route sekali dari representasi prompt setelah G1; contoh ini hanya perbandingan manual.

In [ ]:
route = (1, 0, 1, 0, 1)
inverse_route = tuple(1 - bit for bit in route)

prompt = 'Ringkas artikel berikut dalam satu kalimat: Pemerintah daerah membuka layanan perpustakaan keliling untuk menjangkau sekolah di wilayah terpencil.'
messages = [{'role': 'user', 'content': prompt}]
chat_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(chat_text, return_tensors='pt').to(DEVICE)

def generate_text(max_new_tokens=48):
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print('Model penuh:\n', generate_text())
for label, candidate_route in [('Route awal', route), ('Route inverse', inverse_route)]:
    active = active_layers_for_route(candidate_route)
    skipped = skipped_layers(active)
    print(f'\n{label} G2–G6: {candidate_route}')
    print('Layer aktif (nomor mulai 1):', [i + 1 for i in active])
    print('Layer dilewati (nomor mulai 1):', [i + 1 for i in skipped])
    print(f'Aktif {len(active)} dari {len(layers)} layer')
    with temporarily_skip_layers(skipped):
        routed_text = generate_text()
    print('Keluaran:\n', routed_text)
print('Layer asli dipulihkan:', all(type(layer).__name__ != 'SkipDecoderLayer' for layer in layers))

## 6. Batas demonstrasi
- Layer yang dilewati meneruskan hidden state ke layer berikutnya; bobotnya tidak dihapus.
- Belum ada pooling token artikel, MLP router 1024→64→5, pelatihan, atau target route berbasis NLL.
- Integrasi KV cache dan kesetaraan perilaku perlu diperiksa khusus sebelum implementasi eksperimen utama.
- Skipping tidak menjamin waktu lebih singkat pada semua perangkat; overhead dan panjang keluaran juga berpengaruh. Keluaran di atas hanya demonstrasi, bukan klaim kualitas.